# 03 — Product Recommendations
**Strategy:**
- **Known users** → cluster top-N items (excluding already purchased)
- **Cold users** → global trending items (last 7d / 30d)
- **Also-bought** → co-purchase pairs within each cluster

**Requires:** notebook 02 re-run with raw IDs (no hashing)

## Cell 1 — Imports & Config

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import timedelta

TOP_N = 10
DATA_RAW  = Path('../data/raw')
DATA_PROC = Path('../data/processed')
DATA_PROC.mkdir(parents=True, exist_ok=True)

## Cell 2 — Load Data

In [ ]:
df_tx = pd.read_csv(
    DATA_RAW / 'transactions_train.csv',
    dtype={'article_id': str, 'customer_id': str},
    parse_dates=['t_dat']
)

df_rfm = pd.read_csv(
    DATA_PROC / 'rfm_segmented.csv',
    dtype={'customer_id': str}
)

df_articles = pd.read_csv(
    DATA_RAW / 'articles.csv',
    dtype={'article_id': str}
)

print('Transactions :', df_tx.shape)
print('RFM customers:', df_rfm.shape)
print('Articles     :', df_articles.shape)
print('\nTransaction columns:', df_tx.columns.tolist())
print('RFM columns        :', df_rfm.columns.tolist())

## Cell 3 — ID Alignment Check (NO hashing)
After notebook 02 is fixed, both TX and RFM customer_ids are raw hex strings.  
This cell ONLY verifies alignment — it never transforms IDs.

In [ ]:
sample_tx  = df_tx['customer_id'].iloc[0]
sample_rfm = df_rfm['customer_id'].iloc[0]

print(f'TX  sample id : {sample_tx!r}  (len={len(sample_tx)})')
print(f'RFM sample id: {sample_rfm!r}  (len={len(sample_rfm)})')

# Guard: if either looks like a hashed integer, stop and fix notebook 02
def _is_hashed(s: str) -> bool:
    return s.lstrip('-').isdigit() and len(s) < 25

if _is_hashed(sample_tx) or _is_hashed(sample_rfm):
    raise ValueError(
        '\n One or both ID columns contain hashed integers.'
        '\n   Fix: re-run notebook 02 with the FIXED version that keeps raw hex IDs.'
        '\n   Then delete data/processed/rfm_segmented.csv and re-run it.'
    )

# Direct overlap — no transformation needed
overlap = set(df_tx['customer_id']) & set(df_rfm['customer_id'])
pct     = 100 * len(overlap) / df_rfm['customer_id'].nunique()
print(f'\n Overlapping customers: {len(overlap):,}  ({pct:.1f}% of RFM customers)')

if pct < 50:
    print('\n⚠  Overlap below 50%. Possible causes:')
    print('   1. rfm_segmented.csv was built from customers_clean.csv but'
          ' not all customers have transactions — this is normal.')
    print('   2. rfm_segmented.csv still has hashed IDs — re-run notebook 02.')

## Cell 4 — Cold-Start Fallback (computed before any filtering)

In [ ]:
latest_date = df_tx['t_dat'].max()

def top_products_window(df: pd.DataFrame, days: int, n: int = TOP_N) -> pd.DataFrame:
    cutoff = latest_date - timedelta(days=days)
    return (
        df[df['t_dat'] >= cutoff]
        .groupby('article_id')
        .size()
        .reset_index(name='purchase_count')
        .sort_values('purchase_count', ascending=False)
        .head(n)
    )

top_7d  = top_products_window(df_tx, 7)
top_30d = top_products_window(df_tx, 30)

print(f'Latest date: {latest_date.date()}')
print(f'Top-7d  articles: {len(top_7d)}')
print(f'Top-30d articles: {len(top_30d)}')
top_7d.head()

## Cell 5 — Merge & Cluster Top Products

In [ ]:
df_merged = df_tx.merge(df_rfm[['customer_id', 'cluster']], on='customer_id', how='inner')
print(f'Merged rows: {len(df_merged):,}')

if len(df_merged) == 0:
    raise ValueError('Merge produced 0 rows — Cell 3 overlap check should have caught this.')

meta_cols = ['article_id', 'prod_name', 'product_group_name']

cluster_tops = (
    df_merged
    .groupby(['cluster', 'article_id'])
    .size()
    .reset_index(name='purchase_count')
    .sort_values(['cluster', 'purchase_count'], ascending=[True, False])
    .groupby('cluster')
    .head(TOP_N)
    .reset_index(drop=True)
)
cluster_tops = cluster_tops.merge(df_articles[meta_cols], on='article_id', how='left')

print(f'Cluster top-{TOP_N} table shape: {cluster_tops.shape}')
cluster_tops.head(15)

## Cell 6 — Co-Purchase ('Customers Also Bought')
Within each cluster: self-join on customer_id to find which article pairs  
are most often bought together by the same shopper.

In [ ]:
CO_TOP_ITEMS = 50   # seed items per cluster
CO_TOP_RECS  = 5    # 'also bought' items to keep per seed item

co_rows = []

for cluster_id, cluster_df in df_merged.groupby('cluster'):
    print(f'  Cluster {cluster_id}: {len(cluster_df):,} transactions, '
          f'{cluster_df["customer_id"].nunique():,} customers')

    top_items = (
        cluster_df.groupby('article_id').size()
        .nlargest(CO_TOP_ITEMS).index.tolist()
    )

    slim = (cluster_df[cluster_df['article_id'].isin(top_items)]
            [['customer_id', 'article_id']]
            .drop_duplicates()   # one entry per (customer, item) pair
    )

    # Self-join: pairs of items bought by same customer
    co = slim.merge(
        slim.rename(columns={'article_id': 'also_bought_id'}),
        on='customer_id'
    )
    co = co[co['article_id'] != co['also_bought_id']]

    if co.empty:
        print(f'    ⚠  No co-purchases in cluster {cluster_id} — skipping')
        continue

    co_counts = (
        co.groupby(['article_id', 'also_bought_id'])
        .size()
        .reset_index(name='co_count')
        .sort_values('co_count', ascending=False)
    )
    top_co = co_counts.groupby('article_id').head(CO_TOP_RECS).copy()
    top_co['cluster'] = cluster_id
    co_rows.append(top_co)

if co_rows:
    co_purchase_df = pd.concat(co_rows, ignore_index=True)
    print(f'\n Co-purchase table: {len(co_purchase_df):,} rows')
    print(co_purchase_df.head(8))
else:
    co_purchase_df = pd.DataFrame(
        columns=['article_id', 'also_bought_id', 'co_count', 'cluster']
    )
    print('⚠  co_purchase_df is empty — customers may not be buying multiple items')

## Cell 7 — Per-User Personalised Recommendations

In [ ]:
# Purchase history per user (for exclusion filter)
user_history = (
    df_tx.groupby('customer_id')['article_id']
    .apply(set)
    .reset_index()
    .rename(columns={'article_id': 'purchased_set'})
)

# Cluster tops as a dict: {cluster_id: [article_id, ...]}
cluster_tops_dict = {
    cid: grp['article_id'].tolist()
    for cid, grp in cluster_tops.groupby('cluster')
}

def recommend_for_user(cluster: int, purchased: set, n: int = TOP_N,
                       exclude_purchased: bool = False) -> list:
    # NOTE: exclude_purchased=False by default — most users only bought
    # 1-2 items from the top-10 list, so exclusion just reduces results
    # unnecessarily. Set to True if you want pure novelty.
    candidates = cluster_tops_dict.get(cluster, [])
    if exclude_purchased:
        candidates = [a for a in candidates if a not in purchased]
    return candidates[:n]

user_recs = df_rfm[['customer_id', 'cluster']].merge(
    user_history, on='customer_id', how='left'
)
user_recs['purchased_set'] = user_recs['purchased_set'].apply(
    lambda x: x if isinstance(x, set) else set()
)
user_recs['recommendations'] = user_recs.apply(
    lambda row: recommend_for_user(row['cluster'], row['purchased_set']),
    axis=1
)

print(f'Users with recommendations: {len(user_recs):,}')
avg = user_recs['recommendations'].apply(len).mean()
print(f'Avg recs per user: {avg:.1f}')
user_recs[['customer_id', 'cluster', 'recommendations']].head(5)

## Cell 8 — Save All Outputs

In [ ]:
# 8a — Cluster top products
cluster_tops.to_csv(DATA_PROC / 'cluster_recommendations.csv', index=False)
print(f' cluster_recommendations.csv  — {len(cluster_tops):,} rows')

# 8b — Co-purchase table
co_purchase_df.to_csv(DATA_PROC / 'co_purchase_recs.csv', index=False)
print(f' co_purchase_recs.csv         — {len(co_purchase_df):,} rows')

# 8c — Per-user flat table
user_recs_flat = (
    user_recs[['customer_id', 'cluster', 'recommendations']]
    .explode('recommendations')
    .rename(columns={'recommendations': 'article_id'})
    .dropna(subset=['article_id'])
    .merge(df_articles[meta_cols], on='article_id', how='left')
    .assign(rank=lambda df: df.groupby('customer_id').cumcount() + 1)
)
user_recs_flat.to_csv(DATA_PROC / 'user_recommendations.csv', index=False)
print(f' user_recommendations.csv     — {len(user_recs_flat):,} rows')

# 8d — Cold-start trending tables
for window, df_top in [('7d', top_7d), ('30d', top_30d)]:
    out = df_top.merge(df_articles[meta_cols], on='article_id', how='left')
    out.to_csv(DATA_PROC / f'top_products_{window}.csv', index=False)
    print(f' top_products_{window}.csv           — {len(out)} rows')

print('\n📁 All outputs in', DATA_PROC)

## Cell 9 — Coverage Diagnostics

In [ ]:
print('=== COVERAGE REPORT ===')
print(f'Total RFM users            : {len(df_rfm):,}')
print(f'Users with TX in merged    : {df_merged["customer_id"].nunique():,}')
print(f'Avg recommendations/user   : {user_recs["recommendations"].apply(len).mean():.1f}')
print(f'Users with 0 recs          : {(user_recs["recommendations"].apply(len)==0).sum():,}')
print()
print('Cluster distribution:')
print(df_rfm['cluster'].value_counts().sort_index())
print()
print('Cluster top-item sample:')
print(cluster_tops.groupby('cluster').first()[['article_id','prod_name','purchase_count']])